# Notebook 04 — My Report Runs Out of Memory
## Right-Sizing Your Warehouse

**What you'll learn**: When a complex query "spills to disk," it means your warehouse is too small. Here's how to diagnose it and when to ask for a bigger warehouse.

| | Right Approach | Wrong Approach |
|---|---|---|
| **Sizing** | Match warehouse to query complexity | Use XS for everything (spills to remote storage) |
| **Cost** | Bigger warehouse finishes faster = cheaper overall | Small + slow = same cost, longer wait |

In [ ]:
USE ROLE SYSADMIN;
USE WAREHOUSE HOL_XS;
USE SCHEMA JPMC_PERF_HOL.CONSUMER_BANKING;

ALTER SESSION SET USE_CACHED_RESULT = FALSE;

---
## Step 1: Diagnosing the Problem — "My Fraud Report Takes Forever"

**Scenario**: You're building a monthly fraud loss report — it aggregates 500M rows by account type, channel, and month. On your default XS warehouse, it's painfully slow.

### Run the report on XS (watch it struggle)

In [ ]:
USE WAREHOUSE HOL_XS;

-- Heavy aggregation that WILL spill on XS: per-account monthly summary
-- (25M accounts × 13 months = hundreds of millions of groups)
SELECT
    t.account_id,
    DATE_TRUNC('month', t.transaction_date) AS txn_month,
    COUNT(*) AS txn_count,
    SUM(t.amount) AS total_amount,
    AVG(t.amount) AS avg_amount,
    SUM(CASE WHEN t.fraud_flag THEN t.amount ELSE 0 END) AS fraud_losses
FROM TRANSACTIONS t
WHERE t.transaction_date >= '2024-01-01'
GROUP BY 1, 2
ORDER BY fraud_losses DESC
LIMIT 100;

**Check the Query Profile** → Look for:
- `Bytes spilled to local storage` — the warehouse ran out of memory and is using disk (slow)
- `Bytes spilled to remote storage` — even disk wasn't enough, it's writing to cloud storage (very slow!)

This is like trying to do a year-end reconciliation on a laptop with 4GB RAM — it technically works, but takes 10x longer.

### The Fix: Run the same report on a Medium warehouse (fits in memory)

In [ ]:
USE WAREHOUSE HOL_M;

-- Same query on Medium — enough memory to avoid spilling
SELECT
    t.account_id,
    DATE_TRUNC('month', t.transaction_date) AS txn_month,
    COUNT(*) AS txn_count,
    SUM(t.amount) AS total_amount,
    AVG(t.amount) AS avg_amount,
    SUM(CASE WHEN t.fraud_flag THEN t.amount ELSE 0 END) AS fraud_losses
FROM TRANSACTIONS t
WHERE t.transaction_date >= '2024-01-01'
GROUP BY 1, 2
ORDER BY fraud_losses DESC
LIMIT 100;

**Check the Query Profile** → Compare:
- Zero spilling
- Significantly faster execution
- Credits used: 4x more per hour BUT query finishes 5-10x faster → net cheaper

---
## Step 2: Compare Metrics

In [ ]:
-- Compare XS vs Medium performance
SELECT
    warehouse_name,
    total_elapsed_time / 1000 AS elapsed_sec,
    bytes_scanned / (1024*1024*1024) AS gb_scanned
FROM TABLE(SNOWFLAKE.INFORMATION_SCHEMA.QUERY_HISTORY(
    END_TIME_RANGE_START => DATEADD(minute, -15, CURRENT_TIMESTAMP()),
    END_TIME_RANGE_END => CURRENT_TIMESTAMP(),
    RESULT_LIMIT => 20
))
WHERE query_text ILIKE '%fraud_losses%'
  AND query_type = 'SELECT'
  AND query_text NOT ILIKE '%QUERY_HISTORY%'
ORDER BY start_time DESC
LIMIT 2;

---
## Cleanup

---
## Key Takeaways

| Symptom | Diagnosis | What to Ask For |
|---------|-----------|----------------|
| Query Profile shows "spilling" | Warehouse is too small for this query | *"Can I get a MEDIUM warehouse for my monthly reports?"* |
| Query is fast with zero spilling | Warehouse is right-sized | Don't change anything! |
| Dashboard queries are queued | Too many users on one warehouse | *"Can we enable auto-scaling (multi-cluster)?"* |

**Rule of thumb**: If your query spills → ask for a bigger warehouse. If it doesn't spill on XS → don't upsize.

**💡 What to tell your platform team**:  
*"My monthly aggregation query spills to remote storage on XS. Can I use a MEDIUM warehouse for monthly reporting? Here's the Query Profile showing the spill."*

**Next** → [Notebook 05 — Finding a Needle in 500M Rows](./Notebook_05_Search_Optimization.ipynb)